In [1]:
pip install pandas scikit-learn xgboost imbalanced-learn matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

# ── 1. Load dataset ─
df = pd.read_csv(r'C:\Users\User\Desktop\Theory\creditcard.csv')
print("Dataset shape:", df.shape)
print("Fraud cases:", df['Class'].sum())
print("Fraud ratio:", round(df['Class'].mean() * 100, 4), "%")

Dataset shape: (284807, 31)
Fraud cases: 492
Fraud ratio: 0.1727 %


In [3]:
# ── 2. Data Preprocessing ──

# Drop 'Time' column (not useful for classification)
df = df.drop(columns=['Time'])

# Standardise 'Amount' column to match scale of PCA features
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])

# Separate features and target variable
X = df.drop(columns=['Class'])
y = df['Class']

print("Feature shape:", X.shape)
print("Target distribution:\n", y.value_counts())

# ── 3. Train/Test Split ──
# 80% training, 20% testing with stratification to preserve fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTraining set size:", X_train.shape)
print("Testing set size:", X_test.shape)

# ── 4. Apply SMOTE to training set only ──
# Generate synthetic fraud samples to balance class distribution
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print("Training set distribution:\n", pd.Series(y_train_sm).value_counts())

Feature shape: (284807, 29)
Target distribution:
 Class
0    284315
1       492
Name: count, dtype: int64

Training set size: (227845, 29)
Testing set size: (56962, 29)

After SMOTE:
Training set distribution:
 Class
0    227451
1    227451
Name: count, dtype: int64


In [4]:
# ── 5. Train Models ──

# Model 1: Logistic Regression (baseline model)
print("Training Logistic Regression...")
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sm, y_train_sm)
print("Done.")

# Model 2: Random Forest
print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
print("Done.")

# Model 3: XGBoost
print("Training XGBoost...")
xgb = XGBClassifier(n_estimators=100, random_state=42, 
                    eval_metric='logloss', n_jobs=-1)
xgb.fit(X_train_sm, y_train_sm)
print("Done.")

print("\nAll models trained successfully!")

Training Logistic Regression...
Done.
Training Random Forest...
Done.
Training XGBoost...
Done.

All models trained successfully!


In [7]:
# ── 6. Evaluate Models ──

def evaluate_model(name, model, X_test, y_test):
    # Make predictions on test set
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Print classification report
    print(f"\n{'='*50}")
    print(f"Model: {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, 
          target_names=['Normal', 'Fraud']))
    
    # Calculate AUC-ROC
    auc = roc_auc_score(y_test, y_prob)
    print(f"AUC-ROC Score: {round(auc, 4)}")
    
    return y_pred, auc

# Evaluate all three models
lr_pred, lr_auc = evaluate_model("Logistic Regression", lr, X_test, y_test)
rf_pred, rf_auc = evaluate_model("Random Forest", rf, X_test, y_test)
xgb_pred, xgb_auc = evaluate_model("XGBoost", xgb, X_test, y_test)



Model: Logistic Regression
              precision    recall  f1-score   support

      Normal       1.00      0.97      0.99     56864
       Fraud       0.06      0.92      0.11        98

    accuracy                           0.97     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.97      0.98     56962

AUC-ROC Score: 0.97

Model: Random Forest
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.87      0.83      0.85        98

    accuracy                           1.00     56962
   macro avg       0.94      0.91      0.92     56962
weighted avg       1.00      1.00      1.00     56962

AUC-ROC Score: 0.9737

Model: XGBoost
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.69      0.87      0.77        98

    accuracy                           1.00     56962
   macro avg       0

In [8]:
# Evaluate XGBoost separately
xgb_pred, xgb_auc = evaluate_model("XGBoost", xgb, X_test, y_test)


Model: XGBoost
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.69      0.87      0.77        98

    accuracy                           1.00     56962
   macro avg       0.84      0.93      0.88     56962
weighted avg       1.00      1.00      1.00     56962

AUC-ROC Score: 0.9753


In [ ]:
# ── 7. Comparison Summary Table ──
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Precision': [0.06, 0.87, 0.69],
    'Recall': [0.92, 0.83, 0.87],
    'F1-Score': [0.11, 0.85, 0.77],
    'AUC-ROC': [0.97, 0.9753, 0.9753]
})

print("Model Comparison Summary:")
print(results.to_string(index=False))

# ── 8. Plot comparison chart ──
metrics = ['Precision', 'Recall', 'F1-Score', 'AUC-ROC']
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width, results.iloc[0][1:], width, 
               label='Logistic Regression', color='#FF6B6B')
bars2 = ax.bar(x, results.iloc[1][1:], width, 
               label='Random Forest', color='#4ECDC4')
bars3 = ax.bar(x + width, results.iloc[2][1:], width, 
               label='XGBoost', color='#45B7D1')

ax.set_xlabel('Metrics')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison for Fraud Detection')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1.1)

# Add value labels on bars
for bar in [bars1, bars2, bars3]:
    for rect in bar:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                   xy=(rect.get_x() + rect.get_width()/2, height),
                   xytext=(0, 3), textcoords="offset points",
                   ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Chart saved as model_comparison.png")

# ── 9. Save best model (Random Forest) ──
joblib.dump(rf, 'fraud_detection_model.pkl')
print("Best model (Random Forest) saved as fraud_detection_model.pkl")


NameError: name 'pd' is not defined

In [11]:
import joblib
import os

# Save model
joblib.dump(rf, r'C:\Users\User\fraud_detection_model.pkl')
print("Saved to:", os.path.abspath('C:\\Users\\User\\fraud_detection_model.pkl'))

Saved to: C:\Users\User\fraud_detection_model.pkl


NameError: name 'python' is not defined

In [10]:
pip install flask

Defaulting to user installation because normal site-packages is not writeable
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
Using cached itsdangerous-2.2.0-py3-none-any.whl (16 kB)

   ----- ---------------------------------- 1/7 [itsdangerous]
   ----------- ---------------------------- 2/7 [click]
   ----------- ---------------------------- 2/7 [click]
   ---------------------- ----------------- 4/7 [werkzeug]
   ---------------------- ----------------- 4/7 [werkzeug]
   ---------------------- ----------------- 4/7 [werkzeug]
   ---------------------- ----------------- 4/7 [werkzeug]
   ---------------------- ----------------- 4/7 [werkzeug]
   ---------------------- ----------------- 4/7 [werkzeug]
   ---------------------------- ----------- 5/7 [jinja2]
   ---------------------------- ----------- 5/7 [jinja2]
   ---------------------------- ----------- 5/7 [jinja2]
   ---------------------------------- ----- 6/7 [flask]
   ---------------------------------- -


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
fraud_sample = df[df['Class'] == 1].iloc[0]
features = fraud_sample.drop('Class').tolist()
print(features)

[406.0, -2.3122265423263, 1.95199201064158, -1.60985073229769, 3.9979055875468, -0.522187864667764, -1.42654531920595, -2.53738730624579, 1.39165724829804, -2.77008927719433, -2.77227214465915, 3.20203320709635, -2.89990738849473, -0.595221881324605, -4.28925378244217, 0.389724120274487, -1.14074717980657, -2.83005567450437, -0.0168224681808257, 0.416955705037907, 0.126910559061474, 0.517232370861764, -0.0350493686052974, -0.465211076182388, 0.320198198514526, 0.0445191674731724, 0.177839798284401, 0.261145002567677, -0.143275874698919, 0.0]
